# Carpet diagram — midPliocene-eoi400 temperature benchmarking

Step toward an update of IPCC AR5 WG1 Fig. 9.12 (a *portrait plot* / *carpet
diagram*). For each PMIP midPliocene-eoi400 (mid-Pliocene warm period (~3.2 Ma, eoi400)) simulation we:

1. Load the annual-mean near-surface temperature field `tas_spatialmean_ann` from the
   CVDP output, and the matching `piControl` run.
2. Compute the **midPliocene-eoi400 − piControl anomaly** on each model's native grid.
3. **Sample** that anomaly field at the location of every proxy reconstruction.
4. Summarise the model–data mismatch as the **root-mean-squared error (RMSE)** against
   each reconstruction compilation (Foley-Dowsett SST (Foley & Dowsett 2019; Haywood et al. 2020) and
                      Tierney (PlioDA; Tierney et al. 2024)).

Intermediate tables are written to `output/`. Run with the `my-cli-py` conda env.

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import xarray as xr

ROOT      = os.getcwd()  # run the notebook from the carpet_diagram/ directory
CVDP_DIR  = os.path.join(ROOT, 'cvdp_output_by_experiment')
EXPERIMENT = 'midPliocene-eoi400'
RECON_DIR = os.path.join(ROOT, 'recons', EXPERIMENT)
OUT_DIR   = os.path.join(ROOT, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

VAR        = 'tas_spatialmean_ann'
print('experiment:', EXPERIMENT)
print('CVDP   :', CVDP_DIR)
print('recons :', RECON_DIR)
print('output :', OUT_DIR)

## Step 1 — Reconstruction compilations

Two midPliocene (mid-Piacenzian, ~3.2 Ma; eoi400 boundary conditions) compilations:

- **Foley-Dowsett** — the Foley & Dowsett (2019) alkenone (Uk'37) sea-surface-temperature
  sites, as used by Haywood et al. (2020). The file carries two anomaly columns; following
  Haywood et al. we use the **NOAA ERSST5** anomaly (SST minus the NOAA ERSST5 modern
  climatology), which is the last, unnamed column (`Unnamed: 14`). Scattered sites, so
  every point gets unit weight.
- **Tierney** — PlioDA (Tierney et al. 2024), a data assimilation product. The
  `..._changes.nc` file is the 3.25 Ma minus 0 Ma (modern) `tas_annual` difference, i.e.
  already the right anomaly to compare against `midPliocene-eoi400 − piControl`. The
  180×360 grid is flattened to points (the 3 southernmost and 3 northernmost latitude
  rows are all fill and drop out), and each point gets a `cos(latitude)` weight so the
  near-global RMSE is area-fair rather than pole-heavy.

In [ ]:
df = pd.read_csv(os.path.join(RECON_DIR,
                 'FoleyDowsett2019_cs_mp_sst_data_30k_plus_NOAA.csv'))
# The trailing unnamed column is SST minus the NOAA ERSST5 modern climatology.
anom_col = 'Unnamed: 14'
foley = pd.DataFrame({
    'compilation': 'Foley-Dowsett', 'reference': 'Foley & Dowsett 2019',
    'site': df['Location or site'].astype(str).str.strip(),
    'Proxy': "Uk'37 SST",
    'Latitude': df['Latitude'].astype(float), 'Longitude': df['Longitude'].astype(float),
    'Anom': df[anom_col].astype(float),
})
foley['source_table'] = 'FoleyDowsett2019_cs_mp_sst_data_30k_plus_NOAA.csv'
foley['weight'] = 1.0                     # scattered sites: equal weight

# --- Tierney / PlioDA: flatten the gridded 3.25 Ma - modern tas anomaly to points ---
PLIODA_FILE = 'plioDAMainResults.tas_annual_latePlio_changes.nc'
ds = xr.open_dataset(os.path.join(RECON_DIR, PLIODA_FILE), decode_times=False)
tas = ds['tas_annual'].where(np.abs(ds['tas_annual']) < 1e30)
lon2d, lat2d = np.meshgrid(tas['lon'].values, tas['lat'].values)
tierney = pd.DataFrame({
    'compilation': 'Tierney', 'reference': 'Tierney et al. 2024 (PlioDA)', 'site': np.nan,
    'Proxy': 'PlioDA SAT (assim.)',
    'Latitude': lat2d.ravel().astype(float), 'Longitude': lon2d.ravel().astype(float),
    'Anom': tas.values.ravel().astype(float),
})
tierney['source_table'] = PLIODA_FILE
# Regular lat/lon grid: cos-latitude weight makes the RMSE area-fair.
tierney['weight'] = np.cos(np.deg2rad(tierney['Latitude']))

recon = pd.concat([foley, tierney], ignore_index=True)
recon = recon.dropna(subset=['Latitude', 'Longitude', 'Anom'])

print(recon.groupby('compilation').size())
recon_out = os.path.join(OUT_DIR, f'recon_points_{EXPERIMENT}.csv')
recon.to_csv(recon_out, index=False)
print('wrote', recon_out)
recon.head()

## Step 2 — Model midPliocene-eoi400 − piControl anomalies

Each model writes its CVDP field on its own native grid, so anomalies are computed
per model. We pair every `midPliocene-eoi400` file with the same model's `piControl` file
(the main file is the one named `<model>_<experiment>.cvdp_data.<years>.nc`; auxiliary
per-variable files such as `.siconc.` / `.zos.` / `.monsoon.` / `.tas.indices.` carry a
extra token before the years and are skipped). If the two grids ever differ, the control
is bilinearly regridded onto the midPliocene-eoi400 grid before differencing.

In [ ]:
def model_files(experiment):
    """Map model name -> path for the main CVDP file of an experiment.

    Only `<model>_<experiment>.cvdp_data.<start>-<end>.nc` is the main file; the
    auxiliary per-variable outputs (`.siconc.`, `.zos.`, `.monsoon.`, `.tas.indices.`)
    put an extra token before the year range and hold no `tas_spatialmean_ann`.
    """
    out = {}
    pat = os.path.join(CVDP_DIR, experiment, f'*_{experiment}.cvdp_data.*.nc')
    main = re.compile(rf'^(?P<model>.+)_{re.escape(experiment)}\.cvdp_data\.\d+-\d+\.nc$')
    for f in sorted(glob.glob(pat)):
        m = main.match(os.path.basename(f))
        if m:
            out[m.group('model')] = f
    return out

def has_var(path):
    """True if the CVDP file actually carries the tas field (a few don't)."""
    with xr.open_dataset(path, decode_times=False) as ds:
        return VAR in ds.variables

exp_files = model_files(EXPERIMENT)
pi_files  = model_files('piControl')
paired = sorted(set(exp_files) & set(pi_files))
# Some CVDP files lack tas_spatialmean_ann entirely — drop those models.
models = [m for m in paired if has_var(exp_files[m]) and has_var(pi_files[m])]
print(f'{len(models)} models with both {EXPERIMENT} and piControl (and a tas field):')
print(models)
missing_pi = sorted(set(exp_files) - set(pi_files))
if missing_pi:
    print(f'{EXPERIMENT} models with no piControl (skipped):', missing_pi)
no_var = [m for m in paired if m not in models]
if no_var:
    print(f'models dropped — no {VAR} in CVDP file:', no_var)

In [ ]:
def load_field(path):
    da = xr.open_dataset(path, decode_times=False)[VAR].sortby('lat').sortby('lon')
    # A few CVDP grids (e.g. LOVECLIM piControl) carry duplicate lon values,
    # which break interpolation; keep the first occurrence of each coordinate.
    for dim in ('lat', 'lon'):
        _, idx = np.unique(da[dim].values, return_index=True)
        if len(idx) != da.sizes[dim]:
            da = da.isel({dim: np.sort(idx)})
    return da

anomalies = {}
for m in models:
    exp_field = load_field(exp_files[m])
    pi  = load_field(pi_files[m])
    if exp_field.shape != pi.shape or not (np.allclose(exp_field.lat, pi.lat) and np.allclose(exp_field.lon, pi.lon)):
        pi = pi.interp(lat=exp_field.lat, lon=exp_field.lon)
    anomalies[m] = (exp_field - pi).rename('tas_anom')
    print(f'{m:18s} grid {exp_field.shape}  mean anom {float(anomalies[m].mean()):+.2f} C')

## Step 3 — Sample model anomalies at reconstruction locations

Model longitudes run 0–360°, the proxy longitudes −180–180°, so targets are wrapped to
0–360 and the field is made cyclic in longitude before bilinear interpolation.

Each recon point carries a `weight` used later in the RMSE. Scattered-site compilations
weight every point equally (1.0); gridded near-global reconstructions (Cleator, Osman,
Erb, Tierney) set `weight = cos(latitude)` so the RMSE is area-fair rather than
pole-heavy.

In [ ]:
def sample_points(field, lats, lons):
    """Bilinearly sample a (lat, lon) field at scattered points; lon made cyclic."""
    lon_cyc = np.append(field.lon.values, field.lon.values[0] + 360.0)
    fcyc = xr.concat([field, field.isel(lon=0)], dim='lon').assign_coords(lon=lon_cyc)
    ta = xr.DataArray(np.asarray(lats), dims='point')
    to = xr.DataArray(np.asarray(lons) % 360.0, dims='point')
    return fcyc.interp(lat=ta, lon=to).values

sampled = recon[['compilation', 'reference', 'site', 'Proxy',
                 'Latitude', 'Longitude', 'Anom']].copy()
sampled = sampled.rename(columns={'Anom': 'recon_anom'})
# Optional per-point weight (defaults to equal weighting when Step 1 omits it).
sampled['weight'] = recon['weight'].values if 'weight' in recon.columns else 1.0
for m in models:
    sampled[m] = sample_points(anomalies[m], sampled['Latitude'].values, sampled['Longitude'].values)

sampled_out = os.path.join(OUT_DIR, f'model_anom_at_recon_{EXPERIMENT}.csv')
sampled.to_csv(sampled_out, index=False)
print('wrote', sampled_out, '  shape', sampled.shape)
sampled.head()

## Step 4 — RMSE of each model against each compilation

For every model × compilation we take the model-minus-proxy difference across all proxy
points in that compilation and report the (weight-weighted) RMSE and bias, plus the number
of points contributing (a point off the model grid edge can return NaN). With unit weights
this is the ordinary RMSE; gridded compilations use `cos(latitude)` weights (Step 3).

In [ ]:
records = []
for m in models:
    for comp, grp in sampled.groupby('compilation'):
        diff = grp[m].values - grp['recon_anom'].values
        w = grp['weight'].values
        valid = np.isfinite(diff) & np.isfinite(w)
        n = int(valid.sum())
        if n:
            dv, wv = diff[valid], w[valid]
            rmse = float(np.sqrt(np.sum(wv * dv ** 2) / np.sum(wv)))
            bias = float(np.sum(wv * dv) / np.sum(wv))
        else:
            rmse = bias = np.nan
        records.append({'model': m, 'compilation': comp, 'n_points': n,
                        'rmse': rmse, 'bias': bias})

rmse_long = pd.DataFrame(records)
rmse_wide = rmse_long.pivot(index='model', columns='compilation', values='rmse')
rmse_wide.columns = [f'{c}_RMSE' for c in rmse_wide.columns]

rmse_long.to_csv(os.path.join(OUT_DIR, f'rmse_long_{EXPERIMENT}.csv'), index=False)
rmse_wide.to_csv(os.path.join(OUT_DIR, f'rmse_summary_{EXPERIMENT}.csv'))
print(f'wrote rmse_long_{EXPERIMENT}.csv and rmse_summary_{EXPERIMENT}.csv')
rmse_wide.sort_values(rmse_wide.columns[0])

These RMSE values are the building blocks of the carpet diagram: one column per model,
one row per (period, reconstruction compilation), coloured by RMSE. `carpet_figure.py`
reads every `output/rmse_long_<period>.csv`, so re-running this notebook for a new period
makes its rows appear in the portrait plot automatically.